# BatchNorm、LayerNorm 与 RMSNorm：从张量维度手写

**面试问题：三种归一化分别沿哪些维度统计，训练和推理时有什么差异？**

## 回答主线

1. BatchNorm 对每个特征跨 batch 统计均值方差，依赖批组成并维护 running stats。
2. LayerNorm 对每个样本最后若干特征统计，不依赖其他样本。
3. RMSNorm 只除以均方根而不减均值，计算更简单并保留整体偏移。
4. 三者都有可学习缩放，BatchNorm/LayerNorm 通常还有偏置。
5. 小 batch、变长 padding 和训练/推理模式会显著影响 BatchNorm。
6. 面试实现必须明确 `dim`、epsilon、unbiased=False 和广播形状。

## 真实案例

五条客服请求各有三个 Token、每个 Token 两维特征；前两条来自退款流量，中间两条来自物流流量，最后一条来自账号流量，分布明显不同。我们用 PyTorch 基础算子手写三种归一化，打印统计维度和结果，并把 Batch 从 4 缩到 1 复现 BatchNorm 输出漂移；最后加入 Padding Mask。使用可读的离线教学数据解释机制，指标不能外推为线上收益。

### 输入预览：Batch × Token × Hidden 张量

In [1]:
import torch  # 导入 PyTorch 以手写归一化前向计算。

torch.manual_seed(5)  # 固定教学张量和可学习参数。
hidden = torch.tensor([[[1.0, 2.0], [2.0, 3.0], [3.0, 4.0]], [[1.5, 2.5], [2.5, 3.5], [3.5, 4.5]], [[8.0, 10.0], [9.0, 11.0], [10.0, 12.0]], [[7.5, 9.5], [8.5, 10.5], [9.5, 11.5]], [[4.0, 5.5], [5.0, 6.5], [6.0, 7.5]]], dtype=torch.float32)  # 构造五请求三 Token 两维激活并加入中等尺度账号流量。
request_types = ["退款", "退款", "物流", "物流", "账号"]  # 标记三类流量分布并保证至少五条案例。
print("shape=", tuple(hidden.shape), "语义=[batch, token, hidden]")  # 输出张量布局。
for index, (request_type, tensor) in enumerate(zip(request_types, hidden)):  # 逐请求展示激活。
    print(f"request={index} type={request_type} values={tensor.tolist()}")  # 展示 Batch 内分布差异。

shape= (5, 3, 2) 语义=[batch, token, hidden]
request=0 type=退款 values=[[1.0, 2.0], [2.0, 3.0], [3.0, 4.0]]
request=1 type=退款 values=[[1.5, 2.5], [2.5, 3.5], [3.5, 4.5]]
request=2 type=物流 values=[[8.0, 10.0], [9.0, 11.0], [10.0, 12.0]]
request=3 type=物流 values=[[7.5, 9.5], [8.5, 10.5], [9.5, 11.5]]
request=4 type=账号 values=[[4.0, 5.5], [5.0, 6.5], [6.0, 7.5]]


## Baseline 基线：不归一化时两类流量尺度悬殊

In [2]:
raw_request_means = hidden.mean(dim=(1, 2))  # 计算每个请求整体均值。
raw_feature_means = hidden.mean(dim=(0, 1))  # 计算每个隐藏特征跨 Batch/Token 均值。
raw_feature_stds = hidden.std(dim=(0, 1), unbiased=False)  # 计算总体标准差而非样本标准差。
print("请求级均值：", raw_request_means.tolist())  # 展示退款与物流量级差异。
print("特征级全Batch均值：", raw_feature_means.tolist())  # 展示 BatchNorm 的统计对象。
print("特征级全Batch标准差：", raw_feature_stds.tolist())  # 展示缩放尺度。
print(f"最大请求均值/最小请求均值={raw_request_means.max().item() / raw_request_means.min().item():.2f}x")  # 量化分布偏移。

请求级均值： [2.5, 3.0, 10.0, 9.5, 5.75]
特征级全Batch均值： [5.400000095367432, 6.900000095367432]
特征级全Batch标准差： [3.0342488288879395, 3.4650638103485107]
最大请求均值/最小请求均值=4.00x


### 核心实现：三种归一化的统计维度

In [3]:
def manual_batch_norm(tensor, gamma, beta, epsilon=1e-5):  # 对 [B,T,H] 的每个 H 跨 B,T 归一化。
    mean = tensor.mean(dim=(0, 1), keepdim=True)  # 统计维度是 batch 和 token。
    variance = tensor.var(dim=(0, 1), unbiased=False, keepdim=True)  # 使用总体方差匹配训练 BN。
    normalized = (tensor - mean) / torch.sqrt(variance + epsilon)  # 中心化并缩放。
    return normalized * gamma + beta, mean, variance  # 应用可学习仿射并返回统计量。

def manual_layer_norm(tensor, gamma, beta, epsilon=1e-5):  # 对每个 Token 的最后 H 维归一化。
    mean = tensor.mean(dim=-1, keepdim=True)  # 每个 [batch,token] 独立计算均值。
    variance = tensor.var(dim=-1, unbiased=False, keepdim=True)  # 每个 Token 独立计算方差。
    normalized = (tensor - mean) / torch.sqrt(variance + epsilon)  # 进行中心化和缩放。
    return normalized * gamma + beta, mean, variance  # 返回结果和逐 Token 统计量。

def manual_rms_norm(tensor, gamma, epsilon=1e-5):  # 对每个 Token 最后 H 维做均方根缩放。
    mean_square = tensor.pow(2).mean(dim=-1, keepdim=True)  # 不减均值只计算平方均值。
    normalized = tensor * torch.rsqrt(mean_square + epsilon)  # 乘倒数均方根。
    return normalized * gamma, mean_square  # 应用缩放且没有偏置。

gamma = torch.tensor([1.0, 1.0])  # 使用单位缩放便于观察标准化本身。
beta = torch.tensor([0.0, 0.0])  # 使用零偏置。
bn_output, bn_mean, bn_variance = manual_batch_norm(hidden, gamma, beta)  # 运行 BatchNorm。
ln_output, ln_mean, ln_variance = manual_layer_norm(hidden, gamma, beta)  # 运行 LayerNorm。
rms_output, rms_square = manual_rms_norm(hidden, gamma)  # 运行 RMSNorm。
print("BN mean/var：", bn_mean.flatten().tolist(), bn_variance.flatten().tolist())  # 展示跨请求统计。
print("第0个Token LN mean/var：", ln_mean[0, 0].item(), ln_variance[0, 0].item())  # 展示单 Token 统计。
print("第0个Token RMS mean_square：", rms_square[0, 0].item())  # 展示不中心化统计。

BN mean/var： [5.400000095367432, 6.900000095367432] [9.206666946411133, 12.006667137145996]
第0个Token LN mean/var： 1.5 0.25
第0个Token RMS mean_square： 2.5


## 结果解读：均值、方差与 Batch 依赖性

In [4]:
print("方法     全局均值  全局std  第0请求第0Token输出")  # 输出三种归一化对照表头。
for name, output in [("BatchNorm", bn_output), ("LayerNorm", ln_output), ("RMSNorm", rms_output)]:  # 遍历三种结果。
    print(f"{name:<10} {output.mean().item():>8.4f} {output.std(unbiased=False).item():>8.4f} {output[0, 0].tolist()}")  # 展示整体统计和一个 Token。
single_request = hidden[:1]  # 模拟在线推理 batch size=1 且只有退款请求。
single_bn_output, single_bn_mean, single_bn_variance = manual_batch_norm(single_request, gamma, beta)  # 用当前小 Batch 重新统计 BN。
single_ln_output, _, _ = manual_layer_norm(single_request, gamma, beta)  # 同请求运行 LN。
bn_shift = torch.max(torch.abs(single_bn_output - bn_output[:1])).item()  # 比较同一请求因 Batch 组成改变的 BN 输出。
ln_shift = torch.max(torch.abs(single_ln_output - ln_output[:1])).item()  # 比较 LN 输出。
print(f"同一请求从batch=4变batch=1：BN最大变化={bn_shift:.4f}，LN最大变化={ln_shift:.4f}")  # 展示 BN 依赖其他请求而 LN 不依赖。
print("解读：BN 训练时使用当前 Batch，推理通常使用 running stats；LN/RMSNorm 更适合变长序列和小 Batch。")  # 解释模型选择。

方法     全局均值  全局std  第0请求第0Token输出
BatchNorm   -0.0000   1.0000 [-1.4501110315322876, -1.4141148328781128]
LayerNorm    0.0000   1.0000 [-0.9999799728393555, 0.9999799728393555]
RMSNorm      0.9873   0.1590 [0.6324542760848999, 1.2649085521697998]
同一请求从batch=4变batch=1：BN最大变化=2.0617，LN最大变化=0.0000
解读：BN 训练时使用当前 Batch，推理通常使用 running stats；LN/RMSNorm 更适合变长序列和小 Batch。


## 失败案例：Padding 进入统计量污染有效 Token

In [5]:
padded = hidden[:2].clone()  # 复制两个退款请求构造变长 Batch。
padded[1, 2] = torch.tensor([0.0, 0.0])  # 把第二请求最后 Token 设为 Padding。
valid_mask = torch.tensor([[True, True, True], [True, True, False]])  # 定义有效 Token Mask。
unsafe_mean = padded.mean(dim=(0, 1), keepdim=True)  # 错误地把 Padding 零计入 BN 统计。
valid_values = padded[valid_mask]  # 只抽取五个有效 Token。
safe_mean = valid_values.mean(dim=0, keepdim=True)  # 计算 Mask 感知特征均值。
safe_variance = valid_values.var(dim=0, unbiased=False, keepdim=True)  # 计算 Mask 感知总体方差。
safe_normalized = (padded - safe_mean) / torch.sqrt(safe_variance + 1e-5)  # 用有效统计归一化完整张量。
safe_normalized = torch.where(valid_mask.unsqueeze(-1), safe_normalized, torch.zeros_like(safe_normalized))  # 将 Padding 输出重新置零。
print("含Padding均值：", unsafe_mean.flatten().tolist())  # 展示统计被零值拉低。
print("Mask感知均值：", safe_mean.flatten().tolist())  # 展示只由真实 Token 决定。
print("修正后有效输出：", safe_normalized[valid_mask].tolist(), "Padding输出：", safe_normalized[~valid_mask].tolist())  # 展示 Mask 传播。
print("生产边界：真实实现需明确 SyncBN、running momentum、混合精度统计、分布式 Batch、Pre/Post-Norm、序列 Padding 与融合 Kernel。")  # 总结工程边界。

含Padding均值： [1.6666666269302368, 2.5]
Mask感知均值： [2.0, 3.0]
修正后有效输出： [[-1.4141993522644043, -1.4141993522644043], [0.0, 0.0], [1.4141993522644043, 1.4141993522644043], [-0.7070996761322021, -0.7070996761322021], [0.7070996761322021, 0.7070996761322021]] Padding输出： [[0.0, 0.0]]
生产边界：真实实现需明确 SyncBN、running momentum、混合精度统计、分布式 Batch、Pre/Post-Norm、序列 Padding 与融合 Kernel。


## 回归测试：最后只保护统计维度、Batch 依赖与 Padding

In [6]:
assert torch.allclose(bn_output.mean(dim=(0, 1)), torch.zeros(2), atol=1e-6)  # 验证 BN 每个特征跨 B,T 均值为零。
assert torch.allclose(ln_output.mean(dim=-1), torch.zeros(hidden.shape[0], hidden.shape[1]), atol=1e-6)  # 验证 LN 每个 Token 的 H 维均值为零。
assert not torch.allclose(rms_output.mean(dim=-1), torch.zeros(hidden.shape[0], hidden.shape[1]), atol=1e-3)  # 验证 RMSNorm 不执行中心化。
assert bn_shift > 0.5 and ln_shift < 1e-6  # 验证 BN 依赖 Batch 组成而 LN 不依赖。
assert not torch.allclose(unsafe_mean.flatten(), safe_mean.flatten()) and torch.equal(safe_normalized[~valid_mask], torch.zeros(1, 2))  # 验证 Padding 污染及 Mask 修正。
print("回归测试通过：BN/LN/RMS统计、Batch依赖、LN稳定和Padding Mask均成立。")  # 用少量断言总结归一化合同。

回归测试通过：BN/LN/RMS统计、Batch依赖、LN稳定和Padding Mask均成立。
